# HFSS direct central-difference local gradients

This standalone notebook evaluates **physical coordinate axes** $e_i$, not random high-dimensional directions. For $d$ active parameters the ordinary design has $2d$ HFSS calls per step scale. A center evaluation is shared across all scales, so three scales require $6d+1$ calls (thus **79 calls for 13 active parameters**, with `repeats=1`). `gradient[i]` is always the physical derivative for `PARAM_NAMES[i]`, for example $\partial S11/\partial h1$.

`sigma_vec` describes manufacturing variation and weights susceptibility; the finite-difference step is a separate numerical choice. Central difference is a derivative at `x_center`, whereas an MC weighted-ridge slope is a Gaussian-weighted best-fit slope over a finite multivariate region. Bounds can shrink a symmetric step or require a labelled one-sided formula. Rounding, HFSS noise, and nonlinearity can dominate steps that are too small or too large.

The default three-scale design uses `0.25`, `0.5`, and `1.0` times sigma and evaluates the common center only once. Run the cells from top to bottom. Editing `x_center` in **User settings** is intentionally direct. HFSS is only started when `RUN_HFSS=True`; all numerical post-processing can be repeated from saved CSV files.


In [ ]:
from pathlib import Path
import json
import os
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import lib_config as config
import lib_backbone

%load_ext autoreload
%autoreload 2


In [ ]:
# Existing repository config / Backbone are reused without changing the MC notebooks.
_config = config._loadConfig(Path("./_config.toml"))
app_config = config.initParams(_config, debug=True)
backbone = lib_backbone.Backbone(config=app_config)
base_dir = app_config.env.dir_base
cfg = app_config.hfss

PARAM_NAMES = list(cfg.param_names)
bounds = np.column_stack([cfg.lower_bounds, cfg.upper_bounds]).astype(float)
round_decimals = int(app_config.runtime.round_decimals)

# ===== User settings (x_center may be changed directly) =====
x_center = np.asarray([(lo + hi) / 2 for lo, hi in bounds], dtype=float)
sigma_vec = np.asarray([0.1 if name.startswith("h") or name in {"a", "b", "k"} else 0.02 for name in PARAM_NAMES])
PARAM_UNITS = {name: "mm" for name in PARAM_NAMES}
perturbation_enabled = {name: True for name in PARAM_NAMES}
STEP_SCALES = [0.25, 0.5, 1.0] # three-step convergence check
REFERENCE_STEP_SCALE = 0.5
custom_step = None               # dict keyed by every PARAM_NAME, or shape-(d,) array
repeats = 1
objective_col = "S11"
OBJECTIVE_UNIT = "dB"
INCLUDE_CENTER = True            # one shared center across every step scale
RUN_HFSS = False
run_all_rows = True
DEBUG_N_ROWS = 4
ASYMMETRY_THRESHOLD = 0.25
STABILITY_THRESHOLD = 0.20
MC_WEIGHTED_FIT_CSV = None
OUTPUT_DIR = Path("central_difference_run")

print(f"d={len(PARAM_NAMES)}, active={sum(perturbation_enabled.values())}")


In [ ]:
def _enabled_array(param_names, enabled):
    if isinstance(enabled, dict):
        if set(enabled) != set(param_names):
            missing, extra = set(param_names) - set(enabled), set(enabled) - set(param_names)
            raise ValueError(f"perturbation_enabled keys must match PARAM_NAMES; missing={missing}, extra={extra}")
        return np.asarray([bool(enabled[name]) for name in param_names])
    out = np.asarray(enabled, dtype=bool).reshape(-1)
    if out.size != len(param_names):
        raise ValueError("perturbation_enabled length must equal len(PARAM_NAMES).")
    return out


def _step_array(param_names, sigma, step_scale, custom):
    if custom is None:
        return float(step_scale) * sigma
    if isinstance(custom, dict):
        if set(custom) != set(param_names):
            raise ValueError("custom_step keys must match PARAM_NAMES exactly.")
        return np.asarray([custom[name] for name in param_names], dtype=float)
    out = np.asarray(custom, dtype=float).reshape(-1)
    if out.size != len(param_names):
        raise ValueError("custom_step length must equal len(PARAM_NAMES).")
    return out


def validate_central_difference_inputs(param_names, center, sigma, bounds, enabled, steps):
    center, sigma = np.asarray(center, float).reshape(-1), np.asarray(sigma, float).reshape(-1)
    bounds, steps = np.asarray(bounds, float), np.asarray(steps, float).reshape(-1)
    d = center.size
    if len(param_names) != d: raise ValueError("len(PARAM_NAMES) must equal len(x_center).")
    if sigma.size != d: raise ValueError("len(sigma_vec) must equal len(x_center).")
    if bounds.shape != (d, 2): raise ValueError(f"bounds.shape must be ({d}, 2).")
    active = _enabled_array(param_names, enabled)
    if steps.size != d: raise ValueError("step_vec length must equal len(x_center).")
    if np.any(bounds[:, 0] >= bounds[:, 1]): raise ValueError("Every lower_bound must be < upper_bound.")
    if np.any((center < bounds[:, 0]) | (center > bounds[:, 1])): raise ValueError("Every x_center value must lie inside bounds.")
    if np.any(sigma[active] <= 0): raise ValueError("Every active parameter sigma must be positive.")
    if np.any(steps[active] <= 0): raise ValueError("Every active parameter step must be positive.")
    if not np.all(np.isfinite(center)) or not np.all(np.isfinite(sigma)) or not np.all(np.isfinite(bounds)):
        raise ValueError("center, sigma, and bounds must be finite.")
    return center, sigma, bounds, active, steps


In [ ]:
def build_central_difference_design(param_names, center, sigma, bounds, enabled,
                                    step_scale=0.5, custom_step=None, repeats=1,
                                    round_decimals=6, include_center=False):
    """Build arbitrary-dimensional central/one-sided points; never clips and mislabels."""
    if int(repeats) != repeats or repeats < 1: raise ValueError("repeats must be a positive integer.")
    sigma = np.asarray(sigma, float).reshape(-1)
    requested = _step_array(param_names, sigma, step_scale, custom_step)
    center, sigma, bounds, active, requested = validate_central_difference_inputs(
        param_names, center, sigma, bounds, enabled, requested)
    resolution = 10.0 ** (-round_decimals)
    rounded_center = np.round(center, round_decimals)
    logical = []
    center_needed = bool(include_center)

    for i, name in enumerate(param_names):
        if not active[i]: continue
        left, right = center[i] - bounds[i, 0], bounds[i, 1] - center[i]
        h_sym = min(requested[i], left, right)
        points = []
        if h_sym >= resolution:
            scheme, h = "central", h_sym
            points = [("minus", -1, h), ("plus", 1, h)]
        else:
            # At/tighter than input resolution: use the side with room, preferring 2nd order.
            direction = 1 if right >= left else -1
            room = right if direction == 1 else left
            h = min(requested[i], room / 2.0)
            if h >= resolution:
                scheme = "forward" if direction == 1 else "backward"
                points = [("plus" if direction == 1 else "minus", direction, h),
                          ("plus" if direction == 1 else "minus", 2 * direction, h)]
            else:
                h = min(requested[i], room)
                if h < resolution:
                    raise ValueError(f"{name}: no distinguishable point at HFSS resolution {resolution}.")
                scheme = "forward" if direction == 1 else "backward"
                points = [("plus" if direction == 1 else "minus", direction, h)]
            center_needed = True
        for side, multiplier, h_used in points:
            x = center.copy(); x[i] += multiplier * h_used; x = np.round(x, round_decimals)
            if np.array_equal(x, rounded_center):
                raise ValueError(f"{name}: rounded difference point equals rounded center; increase step.")
            actual = abs(x[i] - rounded_center[i]) / abs(multiplier)
            logical.append(dict(parameter_index=i, parameter_name=name, side=side,
                                requested_step=requested[i], actual_step=actual,
                                offset_multiplier=int(multiplier), difference_scheme=scheme,
                                is_center=False, step_scale=float(step_scale), _x=x))

    if center_needed:
        logical.insert(0, dict(parameter_index=-1, parameter_name="__center__", side="center",
            requested_step=0.0, actual_step=0.0, offset_multiplier=0,
            difference_scheme="fixed", is_center=True, step_scale=float(step_scale), _x=rounded_center))
    keys = [tuple(row["_x"]) for row in logical]
    if len(keys) != len(set(keys)):
        raise ValueError("Duplicate physical points were generated (before repeats).")
    rows=[]
    for point_index, item in enumerate(logical):
        x=item.pop("_x")
        for repeat_index in range(int(repeats)):
            row=dict(item, sample_id=f"s{step_scale:g}_p{point_index:04d}", repeat_index=repeat_index)
            row.update({name: float(x[j]) for j, name in enumerate(param_names)})
            rows.append(row)
    design=pd.DataFrame(rows)
    if design.duplicated(["sample_id", "repeat_index"]).any():
        raise ValueError("Duplicate sample_id/repeat_index keys were generated.")
    return design


def combine_step_designs(designs, share_center=True):
    """Combine scale designs, evaluating an identical center once per repeat."""
    if not designs:
        raise ValueError("At least one step-scale design is required.")
    combined = pd.concat(designs, ignore_index=True)
    center = combined[combined["is_center"]].copy()
    noncenter = combined[~combined["is_center"]].copy()
    if share_center and len(center):
        # All center rows must describe the same physical point. One row per repeat is retained.
        parameter_columns = [c for c in combined.columns if c not in {
            "parameter_index", "parameter_name", "side", "requested_step", "actual_step",
            "offset_multiplier", "difference_scheme", "is_center", "step_scale",
            "sample_id", "repeat_index"
        }]
        if parameter_columns and len(center[parameter_columns].drop_duplicates()) != 1:
            raise ValueError("Step-scale designs do not share the same physical center.")
        center = center.sort_values(["step_scale", "repeat_index"]).drop_duplicates("repeat_index")
        center["sample_id"] = "shared_center"
        center["step_scale"] = np.nan
    out = pd.concat([center, noncenter], ignore_index=True)
    if out.duplicated(["sample_id", "repeat_index"]).any():
        raise ValueError("Duplicate sample_id/repeat_index keys exist after combining scales.")
    return out.sort_values(["is_center", "parameter_index", "step_scale", "side", "repeat_index"],
                           ascending=[False, True, True, True, True], na_position="first").reset_index(drop=True)


In [ ]:
def _detect_frequency_s11_columns(df):
    freq = next((c for c in df if "freq" in str(c).lower()), df.columns[0])
    candidates = [c for c in df if c != freq and ("s(" in str(c).lower() or "s11" in str(c).lower())]
    return freq, candidates[0] if candidates else [c for c in df if c != freq][0]


def evaluate_central_difference_design(design, param_names, backbone, config_raw, app_config,
                                       output_dir, objective_col="S11", run_all_rows=True,
                                       debug_n_rows=4):
    """Evaluate missing rows with the existing Backbone watcher and checkpoint each result."""
    output_dir=Path(output_dir); output_dir.mkdir(parents=True, exist_ok=True)
    result_path=output_dir / "central_difference_results.csv"
    prior=pd.read_csv(result_path) if result_path.exists() else pd.DataFrame()
    completed=set()
    if len(prior) and objective_col in prior:
        ok=np.isfinite(pd.to_numeric(prior[objective_col], errors="coerce"))
        completed=set(map(tuple, prior.loc[ok, ["sample_id", "repeat_index"]].to_numpy()))
    pending=design[~design.apply(lambda r:(r.sample_id,r.repeat_index) in completed, axis=1)]
    if not run_all_rows: pending=pending.head(debug_n_rows)
    model_paths, model_paths_str=backbone._get_path_models()
    temp_file=str(output_dir / Path(config_raw["io"]["filename_temp"]))
    watcher={"n_simulation":int(len(pending)), "n_repeats":1, "WATCH_DIR":str(output_dir),
      "INPUT_FILE":str(output_dir / Path(config_raw["io"]["filename_input"])),
      "MODEL_FILE":model_paths_str, "RESULTS_FILE":str(output_dir / Path(config_raw["io"]["filename_output"])),
      "TEMP_FILE":temp_file, "DONE_FLAG_FILE":str(output_dir / "hfss.done")}
    Path(watcher["DONE_FLAG_FILE"]).unlink(missing_ok=True)
    config_json=app_config.env.dir_base / "_config_HFSS.json"
    with open(config_json,"w") as f: json.dump(watcher,f,indent=2)
    actual=0
    try:
        for _, meta in pending.iterrows():
            record=meta.to_dict()
            try:
                params=meta[param_names].to_numpy(float)
                sim_id=backbone._getSimulationID()
                backbone.call_subroutine(watcher, sim_id, param_names, params,
                                         value_fmt=f"{{:.{app_config.runtime.round_decimals}f}}")
                sweep=pd.read_csv(temp_file); freq_col,s11_col=_detect_frequency_s11_columns(sweep)
                values=sweep[s11_col].to_numpy(float); frequencies=sweep[freq_col].to_numpy(float)
                j=int(np.nanargmin(values)); record[objective_col]=float(values[j])
                record["S11_min_freq_GHz"]=float(frequencies[j]); record["evaluation_status"]="success"
                Path(temp_file).unlink(missing_ok=True)
            except Exception as exc:
                record[objective_col]=np.nan; record["evaluation_status"]="failed"; record["evaluation_error"]=str(exc)
            pd.DataFrame([record]).to_csv(result_path,mode="a",header=not result_path.exists(),index=False)
            actual += 1
    finally:
        Path(watcher["DONE_FLAG_FILE"]).touch(); config_json.unlink(missing_ok=True)
    combined=pd.read_csv(result_path)
    return combined, {"predicted_pending_evaluations":len(pending), "actual_evaluations":actual}


In [ ]:
def aggregate_repeated_results(results, objective_col="S11"):
    if objective_col not in results: raise ValueError(f"HFSS results lack objective_col={objective_col!r}.")
    y=pd.to_numeric(results[objective_col], errors="coerce")
    successful=results.get("evaluation_status", pd.Series("success",index=results.index)).eq("success")
    if not np.all(np.isfinite(y[successful])): raise ValueError("All successful objective values must be finite.")
    good=results.loc[successful & np.isfinite(y)].copy(); good[objective_col]=y[successful & np.isfinite(y)]
    keys=["parameter_index","parameter_name","side","requested_step","actual_step",
          "offset_multiplier","difference_scheme","is_center","step_scale"]
    return good.groupby(keys,dropna=False,sort=False)[objective_col].agg(
        value_mean="mean", value_std=lambda s:s.std(ddof=1) if len(s)>1 else np.nan,
        n_evaluations="size").reset_index()


def compute_central_difference_gradient(aggregated, param_names, sigma, enabled,
                                        objective_unit="dB", param_units=None, epsilon=1e-12):
    active=_enabled_array(param_names,enabled); sigma=np.asarray(sigma,float); param_units=param_units or {}
    center_rows=aggregated[aggregated.side.eq("center")]
    center=float(center_rows.value_mean.mean()) if len(center_rows) else np.nan
    rows=[]
    for i,name in enumerate(param_names):
        unit=param_units.get(name,"unknown"); base=dict(parameter_index=i,parameter_name=name,
          parameter_unit=unit,objective_unit=objective_unit,gradient_unit=(f"{objective_unit} / {unit}" if unit!="unknown" else "unknown"),sigma=sigma[i])
        if not active[i]:
            rows.append(dict(base,gradient=0.0,requested_step=np.nan,actual_step=np.nan,difference_scheme="fixed",
              plus_mean=np.nan,minus_mean=np.nan,center_value=center,plus_std=np.nan,minus_std=np.nan,
              gradient_standard_error=np.nan,forward_slope=np.nan,backward_slope=np.nan,asymmetry_abs=np.nan,
              asymmetry_ratio=np.nan,second_derivative=np.nan,n_plus_evaluations=0,n_minus_evaluations=0,status="fixed",warning="fixed gradient stored as 0; susceptibility contribution is 0")); continue
        a=aggregated[aggregated.parameter_index.eq(i)]
        if a.empty:
            rows.append(dict(base,gradient=np.nan,requested_step=np.nan,actual_step=np.nan,difference_scheme="unknown",
              plus_mean=np.nan,minus_mean=np.nan,center_value=center,plus_std=np.nan,minus_std=np.nan,
              gradient_standard_error=np.nan,forward_slope=np.nan,backward_slope=np.nan,asymmetry_abs=np.nan,
              asymmetry_ratio=np.nan,second_derivative=np.nan,n_plus_evaluations=0,n_minus_evaluations=0,status="failed",warning="no successful HFSS evaluations")); continue
        scheme=a.difference_scheme.iloc[0]; h=float(a.actual_step.iloc[0]); plus=a[a.side.eq("plus")]; minus=a[a.side.eq("minus")]
        def val(frame,mult):
            q=frame[frame.offset_multiplier.eq(mult)]; return float(q.value_mean.iloc[0]) if len(q) else np.nan
        p1,m1,p2,m2=val(plus,1),val(minus,-1),val(plus,2),val(minus,-2)
        if scheme=="central": gradient=(p1-m1)/(2*h)
        elif scheme=="forward" and np.isfinite(p2): gradient=(-3*center+4*p1-p2)/(2*h)
        elif scheme=="forward": gradient=(p1-center)/h
        elif scheme=="backward" and np.isfinite(m2): gradient=(3*center-4*m1+m2)/(2*h)
        else: gradient=(center-m1)/h
        ps=float(plus[plus.offset_multiplier.eq(1)].value_std.iloc[0]) if len(plus[plus.offset_multiplier.eq(1)]) else np.nan
        ms=float(minus[minus.offset_multiplier.eq(-1)].value_std.iloc[0]) if len(minus[minus.offset_multiplier.eq(-1)]) else np.nan
        npv=int(plus.n_evaluations.sum()); nmv=int(minus.n_evaluations.sum())
        se=np.sqrt(ps**2/max(npv,1)+ms**2/max(nmv,1))/(2*h) if scheme=="central" and np.isfinite(ps+ms) else np.nan
        fs=(p1-center)/h if np.isfinite(p1+center) else np.nan; bs=(center-m1)/h if np.isfinite(m1+center) else np.nan
        asym=abs(fs-bs) if np.isfinite(fs+bs) else np.nan
        ratio=asym/max(abs(gradient),epsilon) if np.isfinite(asym) else np.nan
        second=(p1-2*center+m1)/h**2 if np.isfinite(p1+m1+center) else np.nan
        warning="this step range has low local linearity" if np.isfinite(ratio) and ratio>ASYMMETRY_THRESHOLD else ""
        rows.append(dict(base,gradient=gradient,requested_step=float(a.requested_step.iloc[0]),actual_step=h,
          difference_scheme=scheme,plus_mean=p1,minus_mean=m1,center_value=center,plus_std=ps,minus_std=ms,
          gradient_standard_error=se,forward_slope=fs,backward_slope=bs,asymmetry_abs=asym,asymmetry_ratio=ratio,
          second_derivative=second,n_plus_evaluations=npv,n_minus_evaluations=nmv,
          status=("success" if np.isfinite(gradient) else "failed"),
          warning=(warning if np.isfinite(gradient) else "missing successful HFSS side/center evaluation")))
    return pd.DataFrame(rows)


In [ ]:
def compute_gradient_diagnostics(gradient_tables, reference_scale=0.5, epsilon=1e-12):
    """Add cross-step stability; tables maps step scale to definition-ordered results."""
    reference=min(gradient_tables, key=lambda s:abs(float(s)-reference_scale))
    ref=gradient_tables[reference].set_index("parameter_name")["gradient"]
    out={}
    for scale,table in gradient_tables.items(): out[scale]=table.copy()
    all_g=pd.concat({s:t.set_index("parameter_name")["gradient"] for s,t in gradient_tables.items()},axis=1)
    stability=all_g.sub(ref,axis=0).abs().max(axis=1)/ref.abs().clip(lower=epsilon)
    for scale in out:
        out[scale]["step_stability_ratio"]=out[scale].parameter_name.map(stability)
        bad=out[scale].step_stability_ratio>STABILITY_THRESHOLD
        out[scale].loc[bad,"warning"]=(out[scale].loc[bad,"warning"].fillna("")+"; unstable across steps (nonlinearity/noise/rounding/step size)").str.strip("; ")
    return out, all_g


def compute_sigma_weighted_susceptibility(table):
    out=table.copy(); g=out.gradient.fillna(0).to_numpy(float); sigma=out.sigma.to_numpy(float)
    out["S_raw"]=g*g; out["C_sigma"]=(sigma*g)**2
    out["S_raw_normalized"]=out.S_raw/out.S_raw.sum() if out.S_raw.sum()>0 else 0.0
    out["C_sigma_normalized"]=out.C_sigma/out.C_sigma.sum() if out.C_sigma.sum()>0 else 0.0
    return out


def plot_central_difference_results(table, step_gradients=None):
    names=table.parameter_name; fig,axes=plt.subplots(2,2,figsize=(14,9))
    axes[0,0].bar(names,table.gradient,yerr=table.gradient_standard_error); axes[0,0].set_title("Physical gradient")
    axes[0,1].bar(names,table.C_sigma_normalized); axes[0,1].set_title("Sigma-weighted contribution")
    axes[1,0].plot(names,table.forward_slope,"o-",label="forward"); axes[1,0].plot(names,table.backward_slope,"s-",label="backward"); axes[1,0].legend()
    colors=table.status.map({"success":"tab:green","failed":"tab:red","fixed":"tab:gray"}).fillna("tab:red")
    axes[1,1].bar(names,1,color=colors); axes[1,1].set_title("Evaluation status")
    for ax in axes.flat: ax.tick_params(axis="x",rotation=60)
    fig.tight_layout(); plt.show()
    if step_gradients is not None and len(step_gradients.columns)>1:
        step_gradients.T.plot(marker="o",figsize=(10,5)); plt.xlabel("step_scale"); plt.ylabel("gradient"); plt.show()


def compare_mc_weighted_fit(cd, csv_path, epsilon=1e-12):
    mc=pd.read_csv(csv_path).rename(columns={"param":"parameter_name","beta_dS11_dx":"gradient_mc_weighted_fit",
                                             "C_sigma":"C_sigma_mc_weighted_fit"})
    out=cd[["parameter_name","gradient","C_sigma"]].rename(columns={"gradient":"gradient_central_difference","C_sigma":"C_sigma_central_difference"}).merge(mc,on="parameter_name",how="left")
    out["absolute_difference"]=(out.gradient_central_difference-out.gradient_mc_weighted_fit).abs()
    out["relative_difference"]=out.absolute_difference/out.gradient_central_difference.abs().clip(lower=epsilon)
    return out


In [ ]:
# Build/save every requested scale, then deduplicate the identical center across scales.
# For K central scales this gives repeats * (2 * d_active * K + 1) evaluations.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
designs=[]
for scale in STEP_SCALES:
    designs.append(build_central_difference_design(PARAM_NAMES,x_center,sigma_vec,bounds,
        perturbation_enabled,scale,custom_step,repeats,round_decimals,INCLUDE_CENTER))
design_df=combine_step_designs(designs, share_center=True)
design_df.to_csv(OUTPUT_DIR/"central_difference_input.csv",index=False)
predicted_evaluations=len(design_df)
print("Predicted HFSS evaluations:",predicted_evaluations)
display(design_df.head())

if RUN_HFSS:
    results_df, evaluation_counts=evaluate_central_difference_design(design_df,PARAM_NAMES,backbone,_config,app_config,
        OUTPUT_DIR,objective_col,run_all_rows,DEBUG_N_ROWS)
    print("Predicted pending / actual evaluations:",evaluation_counts)
else:
    print("RUN_HFSS=False: design saved; no HFSS call was made.")


In [ ]:
# Post-process either the just-completed run or an existing checkpoint CSV.
results_path=OUTPUT_DIR/"central_difference_results.csv"
if results_path.exists():
    results_df=pd.read_csv(results_path)
    aggregated=aggregate_repeated_results(results_df,objective_col)
    tables={}
    shared_center=aggregated[aggregated.side.eq("center")]
    noncenter_scales=sorted(aggregated.loc[~aggregated.side.eq("center"),"step_scale"].dropna().unique())
    for scale in noncenter_scales:
        # Reattach the one evaluated center to every scale for slope/curvature diagnostics.
        scale_aggregated=pd.concat([shared_center,aggregated[aggregated.step_scale.eq(scale)]],ignore_index=True)
        tables[scale]=compute_central_difference_gradient(scale_aggregated,
            PARAM_NAMES,sigma_vec,perturbation_enabled,OBJECTIVE_UNIT,PARAM_UNITS)
    tables,step_gradients=compute_gradient_diagnostics(tables,REFERENCE_STEP_SCALE)
    reference=min(tables,key=lambda s:abs(float(s)-REFERENCE_STEP_SCALE))
    gradient_df=compute_sigma_weighted_susceptibility(tables[reference])
    gradient_df.to_csv(OUTPUT_DIR/"central_difference_gradient.csv",index=False)
    failures=int(results_df.get("evaluation_status",pd.Series("success",index=results_df.index)).ne("success").sum())
    summary={"parameter_names":PARAM_NAMES,"x_center":x_center.tolist(),"sigma_vec":sigma_vec.tolist(),
      "step_scales":STEP_SCALES,"round_decimals":round_decimals,"repeats":repeats,
      "predicted_evaluations":predicted_evaluations,"actual_rows_saved":len(results_df),
      "successful_rows":len(results_df)-failures,"failed_rows":failures,"objective_col":objective_col}
    with open(OUTPUT_DIR/"central_difference_summary.json","w") as f: json.dump(summary,f,indent=2)
    display(gradient_df) # definition order: gradient[i] <-> PARAM_NAMES[i]
    for row in gradient_df.itertuples(): print(f"∂{objective_col} / ∂{row.parameter_name} = {row.gradient:.8g} {row.gradient_unit}")
    plot_central_difference_results(gradient_df,step_gradients)
    if MC_WEIGHTED_FIT_CSV: display(compare_mc_weighted_fit(gradient_df,MC_WEIGHTED_FIT_CSV))


## Analytic, HFSS-free checks

The test function below is dimension-generic. It checks dimensions 2, 5, 10, and 13, parameter ordering, the usual $2d$ call count, fixed parameters, a boundary one-sided fallback, rounding collisions, multi-step convergence, normalization, and explicit failed-evaluation reporting. The second-order central error should decrease as the step is reduced (until rounding/noise dominates).


In [ ]:
def analytic_f(x):
    x=np.asarray(x,float); return np.sin(x).sum()+0.03*np.sum(x**3)+0.08*np.sum(x[:-1]*x[1:])
def analytic_gradient(x):
    x=np.asarray(x,float); g=np.cos(x)+0.09*x*x
    if len(x)>1: g[:-1]+=0.08*x[1:]; g[1:]+=0.08*x[:-1]
    return g

def run_analytic_tests(dimensions=(2,5,10,13)):
    for d in dimensions:
        names=[f"p{i+1}" for i in range(d)]; center=np.linspace(-.3,.4,d); sigma=np.full(d,.1); b=np.tile([-1.,1.],(d,1))
        design=build_central_difference_design(names,center,sigma,b,{n:True for n in names},.1,None,1,8,False)
        assert len(design)==2*d
        results=design.copy(); results["S11"]=[analytic_f(r[names].to_numpy(float)) for _,r in results.iterrows()]; results["evaluation_status"]="success"
        table=compute_central_difference_gradient(aggregate_repeated_results(results),names,sigma,[True]*d)
        assert table.gradient.to_numpy().shape==(d,) and table.parameter_name.tolist()==names
        np.testing.assert_allclose(table.gradient,analytic_gradient(center),rtol=2e-5,atol=2e-5)
    # fixed point absent
    fixed=build_central_difference_design(["a","b"],[0,0],[.1,.1],[[-1,1],[-1,1]],[True,False],.5,None,1,6,False)
    assert not fixed.parameter_name.eq("b").any()
    # exact bound uses labelled second-order forward points plus center
    edge=build_central_difference_design(["a"],[-1],[.1],[[-1,1]],[True],.5,None,1,6,False)
    assert set(edge.difference_scheme)=={"fixed","forward"} and set(edge.offset_multiplier)=={0,1,2}
    # rounding collision is detected
    try: build_central_difference_design(["a"],[0],[1e-8],[[-1,1]],[True],.5,None,1,6,False)
    except ValueError: pass
    else: raise AssertionError("rounding collision was not detected")
    # central error converges and susceptibility sums to one
    errors=[]
    for scale in [1.,.5,.25]:
        q=build_central_difference_design(["a"],[.2],[.2],[[-1,1]],[True],scale,None,1,10,False)
        q["S11"]=[analytic_f(r[["a"]]) for _,r in q.iterrows()]; q["evaluation_status"]="success"
        t=compute_central_difference_gradient(aggregate_repeated_results(q),["a"],[.2],[True]); errors.append(abs(t.gradient.iloc[0]-analytic_gradient([.2])[0]))
    assert errors[2] < errors[1] < errors[0]
    assert np.isclose(compute_sigma_weighted_susceptibility(t).C_sigma_normalized.sum(),1)
    failed=q.copy(); failed.loc[0,"evaluation_status"]="failed"; failed.loc[0,"S11"]=np.nan
    failed_table=compute_central_difference_gradient(aggregate_repeated_results(failed),["a"],[.2],[True])
    assert failed_table.status.iloc[0]=="failed"
    # Three scales share exactly one center: 6*d + 1 rows for repeats=1.
    d=13; names=[f"p{i+1}" for i in range(d)]; center=np.zeros(d); sigma=np.full(d,.1); b=np.tile([-1.,1.],(d,1))
    three=[build_central_difference_design(names,center,sigma,b,[True]*d,s,None,1,8,True) for s in [.25,.5,1.0]]
    shared=combine_step_designs(three)
    assert len(shared)==6*d+1==79 and shared.is_center.sum()==1
    return "analytic tests passed for d=2,5,10,13; three-scale d=13 count=79"

print(run_analytic_tests())
